# csp_workflow_mp — End-to-End Demo

This notebook walks through the four stages of the formula-to-structure pipeline:

1. **Descriptor** — encode a target formula as a 36-dim periodic descriptor.
2. **Retrieve** — search a template pool for symmetry-compatible structures, ranked by descriptor similarity.
3. **Substitute** — find a valid element mapping from a template onto the target.
4. **(Optional) Relax** — refine geometry with MatterSim (skipped here to keep the demo CPU-friendly).

We use a synthetic 3-template pool (built in-cell) so the notebook is self-contained — no Materials Project download required.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from pymatgen.core import Structure, Lattice

from csp_workflow_mp import (
    SubstitutionEngine,
    TemplatePool,
    compute_periodic_descriptors,
    sg_to_pearson_prefix,
)

## 1. Periodic descriptor

The descriptor concatenates two 18-dim vectors: per-group atom fractions, and per-group mean atomic number. Group indices follow the IUPAC 18-column periodic table.

In [ ]:
target = 'KBr'
desc = compute_periodic_descriptors(target)
print(f'target: {target}')
print(f'descriptor shape: {desc.shape}')
print(f'group-1 fraction (alkali):   {desc[0]:.3f}')
print(f'group-17 fraction (halogen): {desc[16]:.3f}')
print(f'group-1 mean Z:              {desc[18]:.1f}  (K = 19)')
print(f'group-17 mean Z:             {desc[34]:.1f}  (Br = 35)')

## 2. Build a synthetic template pool

In a real run, this comes from `scripts/01_download_mp_data.py` + `02_compute_descriptors.py` (~103 K MP entries). Here we fabricate three rocksalt-type templates so the demo runs without any data dependency.

In [ ]:
demo_dir = Path('_demo_pool')
cif_dir  = demo_dir / 'cifs'
cif_dir.mkdir(parents=True, exist_ok=True)

specs = [
    ('demo-001', 'NaCl', 225, 'cF', 'Na', 'Cl'),
    ('demo-002', 'KCl',  225, 'cF', 'K',  'Cl'),
    ('demo-003', 'MgO',  225, 'cF', 'Mg', 'O'),
]
rows = []
for mid, formula, sg, ps, cation, anion in specs:
    s = Structure(Lattice.cubic(4.2), [cation, anion], [[0,0,0],[0.5,0.5,0.5]])
    s.to(filename=str(cif_dir / f'{mid}.cif'))
    d = compute_periodic_descriptors(formula)
    row = {'material_id': mid, 'formula': formula,
           'space_group': sg, 'pearson_prefix': ps,
           'cif_path': f'{mid}.cif'}
    row.update({f'coef_{i:02d}': d[i-1]    for i in range(1,19)})
    row.update({f'prop_{i:02d}': d[17+i]   for i in range(1,19)})
    rows.append(row)
metadata = pd.DataFrame(rows)
metadata[['material_id','formula','space_group','pearson_prefix']]

In [ ]:
pool = TemplatePool(metadata, cif_root=cif_dir)
print(f'pool size: {len(pool)}')
print(f'expected pearson prefix for SG=225: {sg_to_pearson_prefix(225)}')

## 3. Retrieve symmetry-compatible templates

`TemplatePool.search` filters by `(space_group, pearson_prefix)` first, then ranks remaining candidates by cosine distance in the 36-dim descriptor space.

In [ ]:
hits = pool.search(
    space_group=225,
    pearson_prefix='cF',
    descriptor_vector=desc,
    top_n=3,
)
hits[['material_id','formula','space_group','pearson_prefix','pd_distance']]

## 4. Substitute target elements onto the top template

`SubstitutionEngine.find_substitutions` enumerates feasible element mappings and ranks them. We apply the first feasible mapping to get the predicted structure.

In [ ]:
top = hits.iloc[0]
template_struct = Structure.from_file(str(cif_dir / top['cif_path']))

engine  = SubstitutionEngine()
results = engine.find_substitutions(target, template_struct)
feasible = [r for r in results if r.success]
print(f'feasible mappings: {len(feasible)} of {len(results)}')
for r in feasible:
    print(f"  {r.substitution_dict}")

In [ ]:
pred = engine.apply_substitution(template_struct, feasible[0])
print(f'template : {template_struct.composition.reduced_formula}')
print(f'predicted: {pred.composition.reduced_formula}')
pred

## 5. Next: relaxation

In a production run, the predicted structure is then relaxed with MatterSim + BFGS (see `scripts/05_run_benchmark.py`). The relaxed structure is what gets compared to ground truth via `pymatgen.analysis.structure_matcher.StructureMatcher`.

We skip relaxation here — MatterSim weights are ~1 GB and a CUDA GPU is recommended.

In [ ]:
# Clean up the demo pool directory.
import shutil
shutil.rmtree(demo_dir, ignore_errors=True)